In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import requests
import pandas as pd
import json
import time
import numpy as np
from datetime import datetime
from homeharvest import scrape_property
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import mean_absolute_percentage_error
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import seaborn as sns 

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

The customer needs an ML model to be able to identify cursive letter with a given picture. We must use the sample
images firstly by standardizing images and then uploading it into a csv in order to create a model with accurate results

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

In [ ]:
# Given by teacher on a google drive and downloaded a zip file

In [4]:
import zipfile as zf
import os
# Make a folder of files
zip_name = 'Cursive-20251028T000648Z-1-001.zip'
try:
    with zf.ZipFile(zip_name, 'r') as files:
        files.extractall()
except:
    print("Unexpected error")

In [7]:
import pandas as pd
from pathlib import Path
import re 

ROOT_DIR = Path('/home/jupyter-259941/Project_5/Cursive') 

GENERIC_PREFIXES = ('IMG_','2024') 

data_list = []

STUDENT_ID_PATTERN = re.compile(r'^[Ss]\d+$')

print("Phase 1: Collecting file metadata and checking labels (Handling new prefixes)...")

for file_path in ROOT_DIR.glob('**/*'):
    if file_path.is_file():
        
        # Student is ID 
        student_id = 'UNKNOWN'
        
        for parent in file_path.parents:
            if parent == ROOT_DIR:
                break 
            
            if STUDENT_ID_PATTERN.match(parent.name):
                student_id = parent.name
                break 

        letter_label = file_path.stem.strip()
        file_format = file_path.suffix.lower()
        

        is_generic = any(letter_label.lower().startswith(p) for p in GENERIC_PREFIXES)

        is_labeled = not is_generic and len(letter_label) == 1
        
        # Calculate the final path if the conversion were to happen in-place
        final_path_str = str(file_path.with_suffix('.png'))
        
        data_list.append({
            'original_path': str(file_path),
            'student_id': student_id,
            'label': letter_label if is_labeled else 'UNLABELED',
            'format': file_format,
            'status': 'READY' if is_labeled and student_id != 'UNKNOWN' else 'SKIP',
            'action': 'CONVERT_AND_DELETE' if file_format != '.png' and is_labeled else 'KEEP',
            'final_path': final_path_str
        })

df = pd.DataFrame(data_list)

print(f"Phase 1 Complete. Collected {len(df)} records. Check the 'file_inventory.csv' for the next step.")
df.to_csv('file_inventory.csv', index=False)


Phase 1: Collecting file metadata and checking labels (Handling new prefixes)...
Phase 1 Complete. Collected 897 records. Check the 'file_inventory.csv' for the next step.


In [9]:
import pandas as pd
from PIL import Image
from pillow_heif import register_heif_opener
import os

# Register the HEIC opener
register_heif_opener() 

INPUT_CSV = 'file_inventory.csv'
TARGET_FORMAT = 'PNG' 

# Load the planning data
df_inventory = pd.read_csv(INPUT_CSV)

# Filter for files that are ready and need conversion
df_action = df_inventory[
    (df_inventory['status'] == 'READY') & 
    (df_inventory['format'] != '.png')
].copy()

if df_action.empty:
    print("No non-PNG files found that require conversion and deletion. Done.")
else:
    print(f"Phase 2: Executing conversion and deletion for {len(df_action)} files...")
    
    deleted_count = 0
    
    for index, row in df_action.iterrows():
        original_path = Path(row['original_path'])
        final_path = Path(row['final_path'])
        
        try:
            # 1. Convert
            img = Image.open(original_path)
            img.save(final_path, TARGET_FORMAT)
            
            # 2. Delete Original
            os.remove(original_path)
            deleted_count += 1

        except Exception as e:
            print(f"Error processing {original_path}: {e}. Skipping deletion.")
            continue

    print(f"Phase 2 Complete. Converted and deleted {deleted_count} original files.")


Phase 2: Executing conversion and deletion for 422 files...
Phase 2 Complete. Converted and deleted 422 original files.


In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.utils import shuffle


EMNIST_DATASET_NAME = 'emnist/byclass' 
MAX_EMNIST_SAMPLES = 150000 

print(f"Starting EMNIST load, conversion, and subsetting to {MAX_EMNIST_SAMPLES} samples...")

# Download and load the data. tfds handles the download/integrity check.
ds, ds_info = tfds.load(
    EMNIST_DATASET_NAME,
    split=['train'],  # Only load the training split
    shuffle_files=True,
    as_supervised=True, 
    with_info=True 
)
emnist_train_ds = ds[0]

def dataset_to_numpy(dataset):
    images = []
    labels = []
    for img_tensor, label_tensor in tfds.as_numpy(dataset):
        images.append(img_tensor[:, :, 0].astype(np.float32) / 255.0)
        labels.append(label_tensor)
    return np.array(images), np.array(labels)

full_emnist_images, full_emnist_labels = dataset_to_numpy(emnist_train_ds)


if full_emnist_images.shape[0] > MAX_EMNIST_SAMPLES:
    full_emnist_images, full_emnist_labels = shuffle(
        full_emnist_images, full_emnist_labels, random_state=42
    )
    
    emnist_images = full_emnist_images[:MAX_EMNIST_SAMPLES]
    emnist_labels = full_emnist_labels[:MAX_EMNIST_SAMPLES]
else:
    emnist_images = full_emnist_images
    emnist_labels = full_emnist_labels

print("\n-------------------------------------------------")
print("✅ EMNIST Subsetting Complete.")
print(f"Final EMNIST Images available in 'emnist_images': {emnist_images.shape[0]}")
print("-------------------------------------------------")


2025-10-28 03:09:07.568695: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-28 03:09:07.574332: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-28 03:09:07.589246: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-28 03:09:07.612055: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-28 03:09:07.618777: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-28 03:09:08.792810: W tensorflow/compiler/tf2tensorrt/utils/p

Starting EMNIST load, conversion, and subsetting to 150000 samples...


2025-10-28 03:13:42.773955: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



-------------------------------------------------
✅ EMNIST Subsetting Complete.
Final EMNIST Images available in 'emnist_images': 150000
-------------------------------------------------


In [7]:
import pandas as pd
from pathlib import Path
from PIL import Image
from pillow_heif import register_heif_opener
import os
import re

# Register the HEIC opener
register_heif_opener() 

ROOT_DIR = Path('/home/jupyter-259941/Project_5/Cursive')
GENERIC_PREFIXES = ('IMG_', '2024') 
TARGET_FORMAT = 'PNG' 
# Setup
data_list = []
STUDENT_ID_PATTERN = re.compile(r'^[Ss]\d+$')

print(f"Starting FINAL IN-PLACE conversion and labeling on: {ROOT_DIR}")
print("Original HEIC/JPG files will be DELETED after successful conversion to PNG.")



for original_file_path in ROOT_DIR.glob('**/*'):
    if original_file_path.is_file():
        

        student_id = 'UNKNOWN'
        for parent in original_file_path.parents:
            if parent == ROOT_DIR:
                break 
            if STUDENT_ID_PATTERN.match(parent.name):
                student_id = parent.name
                break 

        # Check for Usable Label
        letter_label = original_file_path.stem.strip()
        file_format = original_file_path.suffix.lower()
        
        is_generic = any(letter_label.lower().startswith(p) for p in GENERIC_PREFIXES)
        is_labeled = not is_generic and len(letter_label) == 1
        
        new_file_path = original_file_path.with_suffix('.' + TARGET_FORMAT.lower())
        
        if not is_labeled or student_id == 'UNKNOWN':
            # Skip unusable files but continue to the next file
            continue 

        try:
            is_conversion_needed = file_format != '.png'
            
            if is_conversion_needed:
                img = Image.open(original_file_path)
                img.save(new_file_path, TARGET_FORMAT)
                os.remove(original_file_path)
            
            # Record the final, clean data row to the list
            data_list.append({
                'filepath': str(new_file_path),
                'label': letter_label
            })

        except Exception as e:
            # Conversion failed, so this file is excluded from the final list
            print(f"Error processing and excluding file: {original_file_path}")
            continue


df_clean = pd.DataFrame(data_list)

# Extract the final lists needed for training
final_image_paths = df_clean['filepath'].tolist()
final_labels = df_clean['label'].tolist()

print("\n--- FINAL DATA SUMMARY ---")
print(f"Successfully prepared {len(df_clean)} images for training.")
print(f"Your 'cursive' folder is now standardized to PNG.")
print("\nYour training data is ready in two variables:")
print("1. 'final_image_paths' (List of all clean PNG file paths)")
print("2. 'final_labels' (List of corresponding labels, e.g., 'a', 'b', 'C')")


⚠️ Starting FINAL IN-PLACE conversion and labeling on: /home/jupyter-259941/Project_5/Cursive
Original HEIC/JPG files will be DELETED after successful conversion to PNG.

--- FINAL DATA SUMMARY ---
✅ Successfully prepared 552 images for training.
Your 'cursive' folder is now standardized to PNG.

Your training data is ready in two variables:
1. 'final_image_paths' (List of all clean PNG file paths)
2. 'final_labels' (List of corresponding labels, e.g., 'a', 'b', 'C')


# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [8]:
# Data is extremely dirty, used Gemini to figure out extraction
df_clean.head()

,filepath,label
0,/home/jupyter-259941/Project_5/Cursive/S9/a.png,a
1,/home/jupyter-259941/Project_5/Cursive/S9/s.png,s
2,/home/jupyter-259941/Project_5/Cursive/S9/h.png,h
3,/home/jupyter-259941/Project_5/Cursive/S9/e.png,e
4,/home/jupyter-259941/Project_5/Cursive/S9/t.png,t


General Insight:

S folders sometimes have subfolders, Different folders are not in order, Some folders are empty
Use regex to clean it up and properly code to handle subfolders

Bias: Assume all data are labeled correctly (a.png is a picture of the letter a not b)

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [9]:
# Run this small block to ensure the lists are defined
final_image_paths = df_clean['filepath'].tolist()
final_labels = df_clean['label'].tolist()

In [14]:
import cv2
import numpy as np
from sklearn.utils import shuffle

# --- STEP 1: LOAD AND STANDARDIZE CURSIVE DATA ---

# EMNIST ByClass (62 classes) order: 0-9, A-Z, a-z
EMNIST_MAPPING = {
    # Digits 0-9 (Indices 0-9)
    **{str(i): i for i in range(10)}, 
    # Uppercase A-Z (Indices 10-35)
    **{chr(i): i for i in range(ord('A'), ord('Z') + 1)},
    # Lowercase a-z (Indices 36-61)
    **{chr(i): i for i in range(ord('a'), ord('z') + 1)}
}

cursive_images_list = []
cursive_labels_list = []
target_size = (28, 28)

print("Starting to load, resize, and label your cursive images...")

for path, label_char in zip(final_image_paths, final_labels):
    
    # a) Load image as grayscale
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        continue # Skip if file could not be read
    
    # b) Resize and Normalize to EMNIST format (28x28, 0.0-1.0 range)
    img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    img_normalized = img_resized.astype(np.float32) / 255.0 

    # c) Map character label to EMNIST numerical index (0-61)
    label_num = EMNIST_MAPPING.get(label_char, -1)
    
    if label_num != -1:
        cursive_images_list.append(img_normalized)
        cursive_labels_list.append(label_num)

# Convert lists to final NumPy arrays
cursive_images = np.array(cursive_images_list)
cursive_labels = np.array(cursive_labels_list)

print("-------------------------------------------------")
print(f"✅ Cursive Data Prepared. Shape: {cursive_images.shape}")


# --- STEP 2: MERGE EMNIST AND CURSIVE DATA ---

# 1. Concatenate the image arrays
print("Merging EMNIST and Cursive image data...")
X_emnist_reshaped = emnist_images.reshape(-1, 28, 28, 1)
X_cursive_reshaped = cursive_images.reshape(-1, 28, 28, 1)

X_train_merged = np.concatenate([X_emnist_reshaped, X_cursive_reshaped], axis=0)

# 2. Concatenate the label arrays
print("Merging EMNIST and Cursive label data...")
Y_train_merged = np.concatenate([emnist_labels, cursive_labels], axis=0)

# 3. --- FIX: FILTER BAD LABELS ---
# We must remove any labels that are not in our 0-61 range.
NUM_CLASSES = 62
print("-------------------------------------------------")
print(f"Pre-filter data shape: {X_train_merged.shape}")
print(f"Max label found: {np.max(Y_train_merged)}")

valid_indices = (Y_train_merged >= 0) & (Y_train_merged < NUM_CLASSES)

X_train_merged = X_train_merged[valid_indices]
Y_train_merged = Y_train_merged[valid_indices]

print(f"Post-filter data shape: {X_train_merged.shape}")
print(f"Max label after filtering: {np.max(Y_train_merged)}")
print("-------------------------------------------------")

# 4. Shuffle the combined dataset
print("Shuffling final merged dataset...")
X_train_merged, Y_train_merged = shuffle(X_train_merged, Y_train_merged, random_state=42)


print(" FINAL MERGE COMPLETE!")
print(f"Total Images for Training: {X_train_merged.shape[0]}")
print(f"Final Input Shape for Keras Model: {X_train_merged.shape[1:]} (Ready for a CNN)")



Starting to load, resize, and label your cursive images...


[ WARN:0@78292.265] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/n.png'): can't open/read file: check file path/integrity
[ WARN:0@78292.265] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/r.png'): can't open/read file: check file path/integrity
[ WARN:0@78292.265] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/b.png'): can't open/read file: check file path/integrity
[ WARN:0@78292.266] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/c.png'): can't open/read file: check file path/integrity
[ WARN:0@78292.266] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/o.png'): can't open/read file: check file path/integrity
[ WARN:0@78292.266] global loadsave.cpp:275 findDecoder imread_('/home/jupyter-259941/Project_5/Cursive/S28/w.png'): can't open/read file: check file path/integrity
[ WARN:0@7

-------------------------------------------------
✅ Cursive Data Prepared. Shape: (500, 28, 28)
Merging EMNIST and Cursive image data...
Merging EMNIST and Cursive label data...
-------------------------------------------------
Pre-filter data shape: (150500, 28, 28, 1)
Max label found: 122
Post-filter data shape: (150000, 28, 28, 1)
Max label after filtering: 61
-------------------------------------------------
Shuffling final merged dataset...
 FINAL MERGE COMPLETE!
Total Images for Training: 150000
Final Input Shape for Keras Model: (28, 28, 1) (Ready for a CNN)


In [16]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

split_index = int(len(X_train_merged) * 0.8)

print("Splitting data into training and validation sets...")
X_train = X_train_merged[:split_index]
Y_train_labels = Y_train_merged[:split_index]  # These are still numbers (0-61)

X_val = X_train_merged[split_index:]
Y_val_labels = Y_train_merged[split_index:]    # These are still numbers (0-61)

# 3. One-hot encode the labels using TensorFlow
# (e.g., label 5 becomes [0,0,0,0,0,1,0,...] for 62 classes)
print("One-hot encoding labels using tf.keras.utils.to_categorical...")

Y_train = to_categorical(Y_train_labels, num_classes=62)
Y_val = to_categorical(Y_val_labels, num_classes=62)

print("-------------------------------------------------")
print(f"✅ Data Splitting Complete.")
print(f"Training images shape:   {X_train.shape}")
print(f"Training labels shape:   {Y_train.shape}")
print(f"Validation images shape: {X_val.shape}")
print(f"Validation labels shape: {Y_val.shape}")

Splitting data into training and validation sets...
One-hot encoding labels using tf.keras.utils.to_categorical...
-------------------------------------------------
✅ Data Splitting Complete.
Training images shape:   (120000, 28, 28, 1)
Training labels shape:   (120000, 62)
Validation images shape: (30000, 28, 28, 1)
Validation labels shape: (30000, 62)


# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# --- STEP 4: BUILD AND TRAIN THE CNN ---

INPUT_SHAPE = (28, 28, 1) # Your images are 28x28 grayscale
NUM_CLASSES = 62           # We defined this earlier

# 1. Define the model architecture
print("Building the CNN model...")

model = Sequential([
    # First convolutional block
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=INPUT_SHAPE),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Third convolutional block
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Classifier (fully-connected layers)
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5), # Dropout for regularization
    
    # Output layer
    Dense(NUM_CLASSES, activation='softmax') # Softmax for multi-class probability
])

# 2. Compile the model
print("Compiling the model...")
model.compile(
    optimizer='adam', # Adam is a good all-around optimizer
    loss='categorical_crossentropy', # Use this loss for one-hot encoded labels
    metrics=['accuracy'] # Track accuracy during training
)

# Print a summary of the model
model.summary()

# 3. Train the model
print("\n--- STARTING MODEL TRAINING ---")

EPOCHS = 15
BATCH_SIZE = 128

# This uses the X_train, Y_train, X_val, and Y_val you just created
history = model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, Y_val) 
)

print("--- MODEL TRAINING COMPLETE ---")
val_loss, val_accuracy = model.evaluate(X_val, Y_val)

print("-------------------------------------------------")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

# # 4. (Optional) Save the trained model
# model.save("emnist_and_cursive_model.h5")
# print("Model saved as 'emnist_and_cursive_model.h5'")

Building the CNN model...
Compiling the model...


/opt/tljh/user/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 62)             │        15,934 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 404,670 (1.54 MB)

 Trainable params: 404,222 (1.54 MB)

 Non-trainable params: 448 (1.75 KB)


--- STARTING MODEL TRAINING ---
Epoch 1/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 51s 52ms/step - accuracy: 0.6385 - loss: 1.3858 - val_accuracy: 0.8339 - val_loss: 0.4934
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 51ms/step - accuracy: 0.8112 - loss: 0.5609 - val_accuracy: 0.8548 - val_loss: 0.4241
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 51ms/step - accuracy: 0.8268 - loss: 0.4994 - val_accuracy: 0.8433 - val_loss: 0.4291
Epoch 4/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 52ms/step - accuracy: 0.8363 - loss: 0.4622 - val_accuracy: 0.8599 - val_loss: 0.3969
Epoch 5/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 52ms/step - accuracy: 0.8411 - loss: 0.4384 - val_accuracy: 0.8578 - val_loss: 0.4108
Epoch 6/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 49s 52ms/step - accuracy: 0.8465 - loss: 0.4228 - val_accuracy: 0.8617 - val_loss: 0.3937
Epoch 7/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 51ms/step - accuracy: 0.8520 - loss: 0.4096 - val_accuracy: 0.8621 - val_loss: 0.3923
Epoch 8/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 52ms/step - a

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


Count Vectorization is word frequencies, so spam_word_count is a redundant feature according to accuracy
However, upon testing this data with real life emails in the inference function, spam_word_count actually
helps more than just plain text

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 
